In [1]:
# OLS Robustness Checks (Clean + Fast)

# --- Libraries ---
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.linear_model import LinearRegression
from ucimlrepo import fetch_ucirepo

# ==========================
# 1. Load + Prepare Data
# ==========================

# Load raw UCI housing dataset (same source as housingmodel.py)
data = fetch_ucirepo(id=555)
df = pd.concat([data.data.features, data.data.targets], axis=1)

# Convert key columns to numeric
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
df['bathrooms'] = pd.to_numeric(df['bathrooms'], errors='coerce')
df['square_feet'] = pd.to_numeric(df['square_feet'], errors='coerce')

# Keep only needed columns at this stage
df = df[['price','bedrooms','bathrooms','square_feet','category','cityname']].dropna()

# Drop extreme outliers for stability
df = df[df['price'] < 10000]

# Define neighborhood = city
df['neighborhood'] = df['cityname']

# =====================================================
# Randomly assign a platform BUT with neighborhood mixes
# (this matches the main pipeline so HHI has 0.33–1 range)
# =====================================================
platforms = ['Craigslist','Zillow','Apartments.com']

neighs = df['neighborhood'].unique()
mixes = np.random.dirichlet([1, 1, 1], size=len(neighs))  # different mix per neighborhood
mix_df = pd.DataFrame(mixes, columns=['p_craigslist','p_zillow','p_apts'])
mix_df['neighborhood'] = neighs

df = df.merge(mix_df, on='neighborhood', how='left')

def sample_platform(row):
    return np.random.choice(
        platforms,
        p=[row['p_craigslist'], row['p_zillow'], row['p_apts']]
    )

df['platform'] = df.apply(sample_platform, axis=1)

# ==========================
# 2. Feature Engineering
# ==========================

# Platform-level price dispersion (std deviation)
platform_dispersion = (
    df.groupby(['platform','neighborhood'])['price']
      .std()
      .reset_index()
)
platform_dispersion.rename(columns={'price':'platform_neigh_std'}, inplace=True)
df = df.merge(platform_dispersion, on=['platform','neighborhood'], how='left')

# Landlord multihoming simulation
df['landlord_id'] = np.random.randint(1000,2000,size=len(df))
df['multihomed'] = (df['landlord_id'] % 7 == 0).astype(int)

# =====================================================
# Platform shares + CORRECT HHI (aggregate first, then merge)
# =====================================================
# counts per neighborhood/platform
counts = (
    df.groupby(['neighborhood','platform'])['price']
      .count()
      .reset_index(name='platform_count')
)

# total per neighborhood
total = (
    df.groupby('neighborhood')['price']
      .count()
      .reset_index(name='total_count')
)

# compute shares on aggregated table
counts = counts.merge(total, on='neighborhood')
counts['platform_share_in_neighborhood'] = (
    counts['platform_count'] / counts['total_count']
)

# HHI per neighborhood
hhi = (
    counts
    .groupby('neighborhood')['platform_share_in_neighborhood']
    .apply(lambda s: (s**2).sum())
    .reset_index(name='platform_concentration_hhi')
)

# merge back to listing level
df = df.merge(
    counts[['neighborhood','platform','platform_count','total_count',
            'platform_share_in_neighborhood']],
    on=['neighborhood','platform']
)
df = df.merge(hhi, on='neighborhood')

# Keep only variables we actually regress on
df = df[['price','bedrooms','bathrooms','square_feet',
         'multihomed','platform_neigh_std','platform_concentration_hhi',
         'platform_share_in_neighborhood',  # keep if you want to compare
         'platform','neighborhood']]

# Drop bad values
df = df.replace([np.inf,-np.inf], np.nan).dropna()
df = df[df['price'] > 0]

# Cap dataset size for speed (max 10k rows)
if df.shape[0] > 10000:
    df = df.sample(n=10000, random_state=42)

print("Final dataset shape for OLS:", df.shape)

# ==========================
# 3. Quick Sanity Check (sklearn regression)
# ==========================

X = df[["bedrooms","bathrooms","square_feet",
        "multihomed","platform_neigh_std","platform_concentration_hhi"]]
y = np.log(df["price"])

linreg = LinearRegression().fit(X, y)

print("\n=== Quick sklearn OLS coefficients (fast sanity check) ===")
for var, coef in zip(X.columns, linreg.coef_):
    print(f"{var}: {coef:.4f}")
print("Intercept:", linreg.intercept_)

# ==========================
# 4. Baseline OLS with Robust SEs (HC3)
# ==========================

formula = ("np.log(price) ~ bedrooms + bathrooms + square_feet "
           "+ multihomed + platform_neigh_std + platform_concentration_hhi")

ols_model = smf.ols(formula=formula, data=df).fit(cov_type="HC3")

print("\n=== Baseline OLS (Robust SEs) ===")
print(ols_model.summary())

# Save main results for Appendix table
key_vars = ["bedrooms","bathrooms","square_feet",
            "multihomed","platform_neigh_std","platform_concentration_hhi"]
results_table = ols_model.summary2().tables[1].loc[key_vars]
results_table.to_latex("ols_baseline_table.tex", float_format="%.4f")
print("\nSaved clean LaTeX table to ols_baseline_table.tex")

# ==========================
# 5. Robustness Checks
# ==========================

# (a) Outlier trimming (drop top/bottom 1% of rents)
df_trim = df[(df["price"] > df["price"].quantile(0.01)) &
             (df["price"] < df["price"].quantile(0.99))]
ols_trim = smf.ols(formula=formula, data=df_trim).fit(cov_type="HC3")
print("\n=== OLS with Outlier Trimming (1% tails) ===")
print(ols_trim.summary())

# (b) Subsample splits (high vs. low rent markets)
median_rent = df["price"].median()
df_high = df[df["price"] >= median_rent]
df_low = df[df["price"] < median_rent]

ols_high = smf.ols(formula=formula, data=df_high).fit(cov_type="HC3")
ols_low = smf.ols(formula=formula, data=df_low).fit(cov_type="HC3")

print("\n=== High Rent Markets OLS ===")
print(ols_high.summary())
print("\n=== Low Rent Markets OLS ===")
print(ols_low.summary())

# (c) Alternative dispersion measure (Coefficient of Variation)
df["platform_neigh_cv"] = df.groupby(["platform","neighborhood"])["price"].transform(
    lambda x: x.std()/x.mean() if x.mean()>0 else 0
)

formula_alt = ("np.log(price) ~ bedrooms + bathrooms + square_feet "
               "+ multihomed + platform_neigh_cv + platform_concentration_hhi")

ols_cv = smf.ols(formula=formula_alt, data=df).fit(cov_type="HC3")
print("\n=== OLS with Alternative Dispersion (CV) ===")
print(ols_cv.summary())


c:\Users\sanvi\anaconda3\Lib\site-packages\ucimlrepo\fetch.py:97: DtypeWarning: Columns (0,5,6,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Final dataset shape for OLS: (10000, 10)

=== Quick sklearn OLS coefficients (fast sanity check) ===
bedrooms: -0.0406
bathrooms: 0.1335
square_feet: 0.0004
multihomed: -0.0152
platform_neigh_std: 0.0007
platform_concentration_hhi: -0.0335
Intercept: 6.483349959205974

=== Baseline OLS (Robust SEs) ===
                            OLS Regression Results                            
Dep. Variable:          np.log(price)   R-squared:                       0.334
Model:                            OLS   Adj. R-squared:                  0.334
Method:                 Least Squares   F-statistic:                     369.6
Date:                Sun, 09 Nov 2025   Prob (F-statistic):               0.00
Time:                        17:23:09   Log-Likelihood:                -3754.0
No. Observations:               10000   AIC:                             7522.
Df Residuals:                    9993   BIC:                             7573.
Df Model:                           6                           

In [2]:
# ==========================
# 1. Baseline OLS
# ==========================
ols_model = smf.ols(formula=formula, data=df).fit(cov_type="HC3")

# Save baseline table
results_table = ols_model.summary2().tables[1].loc[key_vars]
results_table.to_latex("ols_baseline_table.tex", float_format="%.4f")

# ==========================
# 2. Outlier Trimming
# ==========================
ols_trim = smf.ols(formula=formula, data=df_trim).fit(cov_type="HC3")
results_table_trim = ols_trim.summary2().tables[1].loc[key_vars]
results_table_trim.to_latex("ols_trim_table.tex", float_format="%.4f")

# ==========================
# 3. Subsample Splits
# ==========================
ols_high = smf.ols(formula=formula, data=df_high).fit(cov_type="HC3")
ols_low = smf.ols(formula=formula, data=df_low).fit(cov_type="HC3")

# Combine into one DataFrame
subsample_table = pd.DataFrame({
    "High Rent coef": ols_high.params.loc[key_vars],
    "High Rent se": ols_high.bse.loc[key_vars],
    "Low Rent coef": ols_low.params.loc[key_vars],
    "Low Rent se": ols_low.bse.loc[key_vars],
})
subsample_table.to_latex("ols_subsample_table.tex", float_format="%.4f")

# 4. Alternative Dispersion
ols_cv = smf.ols(formula=formula_alt, data=df).fit(cov_type="HC3")
results_table_cv = ols_cv.summary2().tables[1].loc[
    ["bedrooms","bathrooms","square_feet",
     "multihomed","platform_neigh_cv","platform_concentration_hhi"]
]
results_table_cv.to_latex("ols_cv_table.tex", float_format="%.4f")

